In [ ]:
from skipalignments import *
############### ENTER THE LOG PATHS HERE ###############
path_to_road_fines_log = '.../path/to/xes'
path_to_request_for_payment_log = '.../path/to/xes'
path_to_international_declarations_log = '.../path/to/xes'
inspected_log = Logs.ROAD_FINES

############### ENTER THE STOCHASTIC ESTIMATOR HERE ###############
ebi_method = EbiWeights.OCCURANCE

In [ ]:
%load_ext autoreload
%autoreload 2
import pm4py
import statistics
import random

In [ ]:
def update_pair_taus(tree:ProcessTree):
    if isinstance(tree, Tau):
        if tree.parent is not None and len(tree.parent.children) == 2:
            other = tree.parent.children[0]
            if other == tree:
                other = tree.parent.children[1]
            if isinstance(other, Activity):
                # set tau
                tree.name = "TAU_" + other.name
            else:
                tree.name = "TAU_" + other.id
        else:
            tree.name = "TAU_" + str(tree.get_distance_to_root()) + str(random.random())
        return
    elif not isinstance(tree, Activity):
        for c in tree.children:
            update_pair_taus(c)
        return
    else:
        return

In [ ]:
path = None
if inspected_log == Logs.ROAD_FINES:
    path = path_to_road_fines_log
elif inspected_log == Logs.REQUEST_FOR_PAYMENT:
    path = path_to_request_for_payment_log
elif inspected_log == Logs.INTERNATIONAL_DECLARATIONS:
    path = path_to_international_declarations_log
log_rf = pm4py.read_xes(path)

In [ ]:
def generate_tree(activities, prob_sequence=0.25, prob_xor=0.25, prob_and=0.25, prob_loop=0.25, prob_tau=0.4, max_children=4):
    operator_r = random.random()
    num_children = random.randint(2,max_children)
    if operator_r > prob_sequence+prob_xor+prob_and+prob_loop:
        # create a leaf node
        if random.random() < prob_tau or len(activities) == 0:
            # tau
            return Tau(None, 'TAU', 0)
        else:
            # random activity
            return Activity(None, activities[random.randint(0,len(activities)-1)], 100000)
    else:
        children = []
        while len(children) < num_children:
            c = generate_tree([act for act in activities if act not in [x.name for x in children if isinstance(x, Activity)]], prob_sequence/2, prob_xor/2, prob_and/2, prob_loop/2, prob_tau, max_children)
            if isinstance(c, Tau) and sum(isinstance(x, Tau) for x in children) > (0 if operator_r < prob_sequence+prob_xor+prob_and else 1):
                # no two taus in non-loops
                continue
            if isinstance(c, Activity) and sum(isinstance(x, Activity) and x.name == c.name for x in children) > 0:
                # no duplicate label on same leafs
                continue
            children.append(c)
        if operator_r < prob_sequence:
            # sequence node
            node = Sequence(None, children)
        elif operator_r < prob_sequence+prob_xor:
            # xor node
            node = Xor(None, children)
        elif operator_r < prob_sequence+prob_xor+prob_and:
            # and node
            node = And(None, children)
        else:
            # loop node
            children = children[:2]
            node = Loop(None, children)
        for c in children:
            c.set_parent(node)
        return node

In [ ]:
def get_variant_dict(log):
    variants = dict()
    for k,v in pm4py.statistics.variants.log.get.get_variants_from_log_trace_idx(log).items():
        variants[k] = len(v)
    variants = dict(sorted(variants.items(), key=lambda x: -x[1]))
    return variants

In [ ]:
def get_activities(log):
    variants = get_variant_dict(log)
    activities = []
    for var in variants.keys():
        for act in list(var):
            if act not in activities:
                activities.append(act)
    return activities

In [ ]:
activities_rf = get_activities(log_rf)

In [ ]:
seed = None
if inspected_log == Logs.ROAD_FINES:
    seed = 2012
elif inspected_log == Logs.REQUEST_FOR_PAYMENT:
    seed = 2012
elif inspected_log == Logs.INTERNATIONAL_DECLARATIONS:
    seed = 2011 #2004
random.seed(seed)
tree_rf = generate_tree(activities_rf[:len(activities_rf)-len(activities_rf)//4])
tree_rf

In [ ]:
process_tree_rf = tree_rf.to_pm4py()
tree_rf = ProcessTree.from_pm4py(process_tree_rf, 100000, 0, 0)
update_pair_taus(tree_rf)
pm4py.view_process_tree(process_tree_rf, format='png')

In [ ]:
output_path = None
if inspected_log == Logs.ROAD_FINES:
    output_path = "./results/random/rf"
elif inspected_log == Logs.REQUEST_FOR_PAYMENT:
    output_path = "./results/random/payment"
elif inspected_log == Logs.INTERNATIONAL_DECLARATIONS:
    output_path = "./results/random/declarations"

if ebi_method == EbiWeights.OCCURANCE:
    output_path += "/occurance"
elif ebi_method == EbiWeights.UNIFORM:
    output_path += "/uniform"

In [ ]:
derivation = DerivationPipeline(tree_rf, log_rf, pn_log=log_rf, pn_method=ebi_method, sagn_timeout=600)

In [ ]:
smodel_path = "smodel.slpn"
if inspected_log == Logs.ROAD_FINES:
    smodel_path = "./rand_results/rf/occurance/smodel.slpn"
elif inspected_log == Logs.INTERNATIONAL_DECLARATIONS:
    smodel_path = "./rand_results/declarations/occurance/smodel.slpn"
derivation.compute(output_path, slpn_path=smodel_path)

In [ ]:
print(derivation.print_blinded())

In [ ]:
derivation.stats()